# 推理资源账本与真实本机计时

浏览器 Python 实验：按顺序运行，变量在单元之间共享。图表来自当前代码计算。


## 1 · KV 显存账本

K、V 各一份；head 数指 KV heads。这里是估算，不是 GPU 显存采样。

In [ ]:
# The blog provides live charts; standalone Python prints chart data.
if 'display_plot' not in globals():
    def display_plot(x, y, title='', xlabel='', ylabel=''):
        print(title, list(zip(x,y)))

layers, kv_heads, head_dim = 32, 8, 128
bytes_per_element, concurrency = 2, 8
def kv_gib(tokens):
    return 2*layers*kv_heads*head_dim*bytes_per_element*concurrency*tokens/2**30
lengths = [512,1024,2048,4096,8192]
for n in lengths:
    print(n, "tokens =>", kv_gib(n), "GiB")
display_plot(lengths, [kv_gib(n) for n in lengths], "KV estimate", "tokens / request", "GiB")

## 2 · TP 分片真的保持结果吗

这个例子展示沿输入维度分片，局部乘积必须求和。输出维度分片则拼接结果，两者不同。

In [ ]:
x = [2,3,5,7]
W = [[1,2],[3,4],[5,6],[7,8]]
full = [sum(x[i]*W[i][j] for i in range(4)) for j in range(2)]
parts = [[sum(x[i]*W[i][j] for i in shard) for j in range(2)] for shard in ([0,1],[2,3])]
merged = [sum(p[j] for p in parts) for j in range(2)]
print("local contributions:",parts)
print("all-reduce sum:",merged, "reference:",full)
assert merged == full

## 3 · ZeRO 训练状态

16 bytes/参数 是此处混合精度 Adam 的约定：2 权重、2 梯度、4 master weight、8 moments；不含 activation。

In [ ]:
P, ranks = 7_000_000_000, 8
for stage in range(4):
    weights = 2*P/(ranks if stage>=3 else 1)
    gradients = 2*P/(ranks if stage>=2 else 1)
    optimizer = 12*P/(ranks if stage>=1 else 1)
    print("ZeRO",stage,"persistent GiB/rank:",(weights+gradients+optimizer)/2**30)
print("Peak also includes activations, temporary gathered parameters and buffers.")

## 4 · 真实浏览器 CPU 计时

测的是 Python 循环，不是 CUDA、BLAS 或 TPU。固定输入，先预热再重复；修改 n 观察增长。不要拿此结果对比推理框架。

In [ ]:
import time, statistics
n = 24
matrix = [[(i+j)%7/7 for j in range(n)] for i in range(n)]
def matmul():
    return [[sum(matrix[i][k]*matrix[k][j] for k in range(n)) for j in range(n)] for i in range(n)]
matmul()
times = []
for _ in range(12):
    start = time.perf_counter()
    out = matmul()
    times.append((time.perf_counter()-start)*1000)
print("shape:",(n,n),"checksum:",sum(map(sum,out)))
print("median ms:",statistics.median(times),"max ms:",max(times))
display_plot(list(range(1,13)),times,"Measured Python matmul","repeat","ms")

## 5 · cache 后的 mask 要从哪里开始

旧 cache 有 4 个 token，新 chunk 有 2 个。先预测每个 query 能看哪些 key，再改 past_length/new_length 重跑。这里构造的是绝对位置的可见性，不含 padding。

In [ ]:
past_length, new_length = 4, 2
assert past_length >= 0 and new_length > 0
allowed = [[int(k <= past_length+q) for k in range(past_length+new_length)] for q in range(new_length)]
wrong = [[int(k <= q) for k in range(past_length+new_length)] for q in range(new_length)]
for q, (correct_row, wrong_row) in enumerate(zip(allowed, wrong)):
    print("absolute query",past_length+q,"correct:",correct_row,"unshifted:",wrong_row)
assert all(sum(row) == past_length+q+1 for q,row in enumerate(allowed))
display_plot(list(range(past_length,past_length+new_length)),[sum(row) for row in allowed],"Visible keys per new query","absolute query position","visible keys")

## 6 · 真正重用 K/V，再与完整重算比较

标准库实现单头注意力，投影使用固定标量、softmax 稳定化。逐 token 的输出必须与完整因果计算相等。它隔离 attention 机制；完整两层 Decoder、位置 embedding、FP32/FP64 与 CPU 计时在另一个标签。

In [ ]:
import math
vectors = [[math.sin(i+j*.7) for j in range(4)] for i in range(8)]
def project(vector, scale):
    return [scale*x for x in vector]
def read_query(q, keys, values):
    scores = [sum(x*y for x,y in zip(q,k))/len(q)**.5 for k in keys]
    unnormalized = [math.exp(s-max(scores)) for s in scores]
    weights = [s/sum(unnormalized) for s in unnormalized]
    return [sum(a*v[d] for a,v in zip(weights,values)) for d in range(len(q))]
full = []
for i,x in enumerate(vectors):
    full.append(read_query(project(x,.7),[project(v,1.1) for v in vectors[:i+1]],[project(v,.9) for v in vectors[:i+1]]))
keys, values, cached = [], [], []
for x in vectors:
    keys.append(project(x,1.1))
    values.append(project(x,.9))
    cached.append(read_query(project(x,.7),keys,values))
errors = [max(abs(a-b) for a,b in zip(expected,actual)) for expected,actual in zip(full,cached)]
assert max(errors) < 1e-12
wrong_output = read_query(project(vectors[-1],.7),keys[:1],values[:1])
wrong_error = max(abs(a-b) for a,b in zip(full[-1],wrong_output))
assert wrong_error > 1e-3
print("cached K/V shape:",(len(keys),len(keys[0])))
print("max correct error:",max(errors),"unshifted-mask error:",wrong_error)
print("At query length 1, a top-left triangle can expose only key 0.")
display_plot(list(range(1,len(vectors)+1)),errors,"Full vs cached attention error","consumed tokens","max absolute error")